<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 9B: *Fire Spread Feature Ablation*
##### Version Number: 4.0
---
### Contents
> *Build Models*\
> *SHAP*\
> *Set Ablation*
---
### Notes
---
### Inputs
---
### Outputs 
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core libraries
import pandas as pd
import numpy as np
import json

# Modeling libraries
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import train_test_split

---

###  Load Data

In [3]:
X_spread = pd.read_csv('../data/processed/X_spread.csv')
y_spread = pd.read_csv('../data/processed/y_spread.csv').squeeze()  # Load as Series
details_spread = pd.read_csv('../data/processed/details_spread.csv')

pal_details = pd.read_csv('../data/processed/pal_details.csv')
pal_X = pd.read_csv('../data/processed/pal_X.csv')
pal_y = pd.read_csv('../data/processed/pal_y.csv')

best_strategy = pd.read_csv('../data/processed/spread_best_strategy.csv')

with open('model_parameters_ignition.json', 'r') as f:
    model_parameters = json.load(f)
    
with open('feature_sets.json', 'r') as f:
    feature_sets = json.load(f)

## Subset

In [4]:
reform = pd.concat([X_spread,y_spread], axis=1)
subset = subset_df(reform, 'Target_Spread_Lag', 500)

y = subset['Target_Spread_Lag']
X = subset.drop(columns='Target_Spread_Lag')

In [5]:
X_rf, y_rf = apply_balancing('RF', best_strategy, X, y)
X_xgb, y_xgb = apply_balancing('XGB', best_strategy, X, y)

## Build Models

In [6]:
RF_parameters = model_parameters['Random Forest']
XGB_parameters = model_parameters['XGBoost']

# Build tuned models
spread_xgb = xgb.XGBClassifier(**XGB_parameters)
spread_rf = RandomForestClassifier(**RF_parameters)

display(RF_parameters)
display(XGB_parameters)

{'n_estimators': 50,
 'max_depth': 10,
 'min_samples_split': 5,
 'max_features': 'log2',
 'class_weight': 'balanced'}

{'objective': 'multi:softmax',
 'num_class': 3,
 'n_estimators': 150,
 'max_depth': 5,
 'learning_rate': 0.1,
 'verbosity': 0}

In [7]:
models = {
    "XGB": spread_xgb, 
    "RF": spread_rf,
}

## SHAP

### XGB Spread SHAP

In [8]:
xgb_top_10 = get_shap(spread_xgb, X_xgb, y_xgb)
xgb_top_10

 24%|=====               | 607/2500 [00:11<00:34]       

 27%|=====               | 664/2500 [00:12<00:33]       

 29%|======              | 723/2500 [00:13<00:31]       

 31%|======              | 782/2500 [00:14<00:30]       

 34%|=======             | 841/2500 [00:15<00:29]       

...

 92%|==================  | 2305/2500 [00:40<00:03]       

 94%|=================== | 2352/2500 [00:41<00:02]       

 96%|=================== | 2406/2500 [00:42<00:01]       

 99%|===================| 2466/2500 [00:43<00:00]       

,0,1
0,other_percent,0.484513
1,1000-hour Dead Fuel Moisture,0.470892
2,Solar Radiation,0.325849
3,population_density,0.274755
4,Season_Spring,0.272993
5,SPEI 90-Day,0.233194
6,Season_Fall,0.226557
7,NDVI_mean_difference,0.224699
8,Solar Radiation 7 Day Avg,0.223208
9,slope_mean,0.217802


### RF Spread SHAP

In [9]:
rf_top_10 = get_shap(spread_rf, X_rf, y_rf)
rf_top_10 

 68%|==============      | 1708/2500 [00:11<00:05]       

 75%|===============     | 1876/2500 [00:12<00:03]       

 81%|================    | 2037/2500 [00:13<00:02]       

 87%|=================   | 2168/2500 [00:14<00:02]       

 93%|=================== | 2321/2500 [00:15<00:01]       

 99%|===================| 2485/2500 [00:16<00:00]       

,0,1
0,slope_mean,0.015129
1,elevation_std,0.014875
2,SPEI 90-Day,0.013700
3,slope_range,0.011066
4,dominant_province_percent,0.010646
5,other_percent,0.010478
6,influence_zone,0.010016
7,slope_max,0.009713
8,forest_percent,0.009655
9,road_density_x_forest_percent,0.008442


## Set Ablation

In [10]:
for set_name, columns in feature_sets.items():
    print(f"{set_name}: {columns}")
    print("\n")

Water Demand: ['Actual Evapotranspiration', 'Solar Radiation', 'Solar Radiation 7 Day Avg', 'Daily Minimum Air Temperature', 'Daily Maximum Air Temperature', 'Daily Maximum Air Temperature 7 Day Avg', 'Daily Minimum Air Temperature 7 Day Avg', 'Vapor Pressure Deficit', 'Vapor Pressure Deficit 7 Day Avg', 'Wind Speed', 'Wind Speed 7 Day Avg']


Water Supply: ['Precipitation', 'Precipitation 7 Day Avg', 'Maximum Relative Humidity', 'Minimum Relative Humidity', 'Maximum Relative Humidity 7 Day Avg', 'Minimum Relative Humidity 7 Day Avg', 'Specific Humidity', '100-hour Dead Fuel Moisture', '1000-hour Dead Fuel Moisture']


Water Supply Indexes: ['SPI 30-Day', 'SPI 180-Day', 'SPEI 30-Day', 'SPEI 90-Day', 'SPEI 180-Day', 'Palmer Drought Severity Index']


Fire Danger: ['Burning Index', 'Energy Release Component', 'Santa_Ana_Score']


Social: ['total_housing', 'total_population', 'housing_density', 'population_density', 'median_income']


Infrastructure: ['power_line_meters', 'power_line_dens

In [11]:
compare = []

compare.append(compare_model(spread_xgb, X, y, best_strategy,
                                     name = 'XGB', test_set = 'Full'))

compare.append(compare_model(spread_rf, X, y, best_strategy,
                                     name = 'RF', test_set = 'Full'))

for set_name, set_list in feature_sets.items():
    for model_name, model in models.items():
        print("Testing " + f"{model_name}: {set_name}")
        X_section = X.drop(columns = set_list)
        compare.append(compare_model(model, X_section, y, best_strategy,
                                     name = model_name, test_set = set_name))

Testing XGB: Water Demand


Testing RF: Water Demand


Testing XGB: Water Supply


Testing RF: Water Supply


Testing XGB: Water Supply Indexes


...

Testing RF: Others


Testing XGB: Coded Ecoregions


Testing RF: Coded Ecoregions


Testing XGB: Coded Seasons


Testing RF: Coded Seasons


In [12]:
comparisons = pd.DataFrame(compare)

In [13]:
XGB = comparisons[comparisons['Model'] == 'XGB'] 
RF = comparisons[comparisons['Model'] == 'RF'] 

In [14]:
full_XGB = XGB.loc[XGB['Test Set']=='Full','Macro F1'].values[0]
full_RF = RF.loc[RF['Test Set']=='Full','Macro F1'].values[0]

In [15]:
XGB['Macro F1 Percent Difference'] = ((full_XGB - XGB['Macro F1']) / full_XGB) * 100
RF['Macro F1 Percent Difference'] = ((full_RF - RF['Macro F1']) / full_RF) * 100

C:\Users\dusti\AppData\Local\Temp\ipykernel_12232\2524239291.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  XGB['Macro F1 Percent Difference'] = ((full_XGB - XGB['Macro F1']) / full_XGB) * 100
C:\Users\dusti\AppData\Local\Temp\ipykernel_12232\2524239291.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  RF['Macro F1 Percent Difference'] = ((full_RF - RF['Macro F1']) / full_RF) * 100


In [16]:
RF

,Test Set,Model,Weighted F1,Macro F1,High Risk Recall,Macro F1 Percent Difference
1,Full,RF,0.681219,0.675160,0.583333,0.000000
3,Water Demand,RF,0.677405,0.673168,0.562500,0.294999
5,Water Supply,RF,0.696096,0.691587,0.635417,-2.433057
7,Water Supply Indexes,RF,0.655316,0.648477,0.593750,3.952003
9,Fire Danger,RF,0.696650,0.692449,0.583333,-2.560849
11,Social,RF,0.669149,0.662505,0.562500,1.874355
13,Infrastructure,RF,0.684944,0.679637,0.541667,-0.663212
15,Elevation,RF,0.662007,0.655382,0.572917,2.929317
17,WUI,RF,0.667185,0.662912,0.583333,1.814034
19,Ecoregion,RF,0.681719,0.676972,0.593750,-0.268451


In [17]:
XGB

,Test Set,Model,Weighted F1,Macro F1,High Risk Recall,Macro F1 Percent Difference
0,Full,XGB,0.722712,0.719102,0.583333,0.000000
2,Water Demand,XGB,0.726229,0.721898,0.572917,-0.388847
4,Water Supply,XGB,0.738148,0.735381,0.635417,-2.263826
6,Water Supply Indexes,XGB,0.661909,0.658611,0.531250,8.412013
8,Fire Danger,XGB,0.714727,0.710343,0.583333,1.218079
10,Social,XGB,0.733378,0.728774,0.614583,-1.344987
12,Infrastructure,XGB,0.727481,0.722298,0.572917,-0.444443
14,Elevation,XGB,0.702664,0.700760,0.583333,2.550644
16,WUI,XGB,0.735515,0.731397,0.625000,-1.709798
18,Ecoregion,XGB,0.719196,0.714155,0.593750,0.687896
